# 04 - Clustering: Unsupervised Anomaly Detection

**DSR Pipeline Phase: Modeling**

This notebook applies 4 unsupervised anomaly detection methods:
1. **KMeans** (Baseline) -- partition-based, assumes k=2 clusters
2. **DBSCAN** (Density) -- density-based, finds clusters of arbitrary shape
3. **HDBSCAN** (Hierarchical Density) -- adaptive density, handles varying densities
4. **Autoencoder** (Deep Learning) -- reconstruction error as anomaly score

All models are trained **without labels** (unsupervised). AAMI labels are used
only for evaluation in the next notebook.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ecg_anomaly.config import SystemConfig
from ecg_anomaly.data.loader import MITBIHLoader
from ecg_anomaly.preprocessing.pipeline import PreprocessingPipeline
from ecg_anomaly.features.signal_pca import SignalPCAExtractor
from ecg_anomaly.models.factory import DetectorFactory
from ecg_anomaly.models.kmeans import KMeansDetector
from ecg_anomaly.models.dbscan import DBSCANDetector
from ecg_anomaly.models.hdbscan_model import HDBSCANDetector
from ecg_anomaly.models.autoencoder import AutoencoderDetector
from ecg_anomaly.visualization.clusters import plot_pca_scatter, plot_anomaly_distribution

sns.set_style("whitegrid")
config = SystemConfig()
config.setup_logging()

## 1. Prepare Data

In [ ]:
# Load, preprocess, extract features
loader = MITBIHLoader(config)
dataset = loader.load(config.dataset_path)

pipeline = PreprocessingPipeline(config)
preprocessed = pipeline.run(dataset)

pca_extractor = SignalPCAExtractor(variance_threshold=config.pca_variance_threshold)
X_pca = pca_extractor.fit_transform(preprocessed.segments)
X_autoencoder = pca_extractor.get_raw_for_autoencoder(preprocessed.segments)
true_labels = preprocessed.labels

print(f"Clustering input (PCA): {X_pca.shape}")
print(f"Autoencoder input (raw scaled): {X_autoencoder.shape}")
print(f"True labels: Normal={int(np.sum(true_labels==0)):,}, Anomalous={int(np.sum(true_labels==1)):,}")

## 2. Available Detectors

In [ ]:
print("Registered detectors:", DetectorFactory.list_detectors())
print()
print("Model configurations:")
for name in config.models:
    params = getattr(config, f"{name}_params")
    print(f"  {name}: {params}")

## 3. Run All 4 Models

Each model is created via the `DetectorFactory` and trained on the unsupervised data.

In [ ]:
detectors = {}

for model_name in config.models:
    params = getattr(config, f"{model_name}_params")
    detector = DetectorFactory.create(model_name, params)

    # Autoencoder uses full-dimensional data; clustering models use PCA
    X = X_autoencoder if model_name == "autoencoder" else X_pca

    print(f"\nFitting {model_name}...")
    detector.fit(X)

    n_anomalies = int(np.sum(detector.anomaly_labels_ == 1)) if detector.anomaly_labels_ is not None else 0
    total = len(X)
    print(f"  Anomalies detected: {n_anomalies:,} / {total:,} ({n_anomalies/total*100:.1f}%)")

    if detector.labels_ is not None:
        unique_labels = set(detector.labels_)
        print(f"  Clusters found: {len(unique_labels - {-1})} (noise points: {int(np.sum(detector.labels_ == -1)):,})")

    detectors[model_name] = detector

## 4. Visualize Clusters in PCA Space

For each model, we plot the first two PCA components colored by:
- Cluster assignment (left)
- Anomaly prediction (right)

In [ ]:
fig, axes = plt.subplots(len(config.models), 2, figsize=(16, 5 * len(config.models)))

for row, model_name in enumerate(config.models):
    detector = detectors[model_name]

    # Left: cluster labels
    ax_left = axes[row, 0]
    if detector.labels_ is not None:
        unique_labels = sorted(set(detector.labels_))
        palette = sns.color_palette("husl", len(unique_labels))
        for idx, label in enumerate(unique_labels):
            mask = detector.labels_ == label
            name = "Noise" if label == -1 else f"Cluster {label}"
            ax_left.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[palette[idx]],
                           alpha=0.3, s=5, label=name)
        ax_left.legend(fontsize=7, markerscale=2)
    else:
        ax_left.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.2, s=5, c="gray")
        ax_left.text(0.5, 0.5, "No cluster labels\n(reconstruction-based)",
                     transform=ax_left.transAxes, ha="center", va="center")
    ax_left.set_title(f"{model_name.upper()} - Cluster Assignment")
    ax_left.set_xlabel("PC1")
    ax_left.set_ylabel("PC2")

    # Right: anomaly predictions
    ax_right = axes[row, 1]
    if detector.anomaly_labels_ is not None:
        normal_mask = detector.anomaly_labels_ == 0
        anomaly_mask = detector.anomaly_labels_ == 1
        ax_right.scatter(X_pca[normal_mask, 0], X_pca[normal_mask, 1],
                        alpha=0.2, s=5, c="#2E86AB", label="Normal")
        ax_right.scatter(X_pca[anomaly_mask, 0], X_pca[anomaly_mask, 1],
                        alpha=0.3, s=5, c="#E74C3C", label="Anomaly")
        ax_right.legend(fontsize=8, markerscale=2)
    ax_right.set_title(f"{model_name.upper()} - Anomaly Prediction")
    ax_right.set_xlabel("PC1")
    ax_right.set_ylabel("PC2")

plt.suptitle("Clustering Results in PCA Space", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Anomaly Distribution Comparison

Compare ground truth vs each model's anomaly assignments.

In [ ]:
for model_name in config.models:
    detector = detectors[model_name]
    if detector.anomaly_labels_ is not None:
        plot_anomaly_distribution(
            true_labels,
            detector.anomaly_labels_,
            model_name=model_name,
        )

## 6. Quick Agreement Check

How much do the models agree with each other?

In [ ]:
# Build agreement matrix
model_names = [m for m in config.models if detectors[m].anomaly_labels_ is not None]
n_models = len(model_names)

agreement = np.zeros((n_models, n_models))
for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        pred1 = detectors[m1].anomaly_labels_
        pred2 = detectors[m2].anomaly_labels_
        agreement[i, j] = np.mean(pred1 == pred2)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(agreement, annot=True, fmt=".3f", cmap="YlGnBu",
            xticklabels=model_names, yticklabels=model_names, ax=ax)
ax.set_title("Pairwise Agreement Between Models")
plt.tight_layout()
plt.show()

## Summary

All 4 models produce anomaly predictions without using labels:
- **KMeans:** Assigns the smaller cluster as anomalous
- **DBSCAN:** Labels noise points (no dense neighborhood) as anomalous
- **HDBSCAN:** Similar density-based approach with adaptive thresholds
- **Autoencoder:** High reconstruction error = anomalous

Each model captures different aspects of the data structure.
The agreement matrix shows how much they overlap in their predictions.

**Next:** `05_evaluation.ipynb` -- formal comparative evaluation with AAMI ground truth.